# What Has the World Model Learned?

World models compress raw pixels into latent vectors — but what information
do those vectors actually contain? **Linear probing** answers this:
train a simple linear model to predict known properties (position, angle, velocity)
from the latent vector. High accuracy = the model learned that property.

This notebook:
1. Generates a dataset with known ground-truth properties
2. Trains a world model on it
3. Probes the latent space for each property
4. Visualizes the latent space with PCA and t-SNE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DilpreetBansi/worldkit/blob/main/notebooks/04_latent_probing.ipynb)

**Time to run:** ~8 minutes on Colab T4

In [ ]:
!pip install "worldkit[train]" scikit-learn -q

## 1. Generate Data with Known Properties

We'll create a simple 2D world: a colored circle at position `(x, y)`
with a known `size` and `color_hue`. Since we control the data generation,
we know exactly what properties are present in each frame.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import csv
from pathlib import Path


def hsv_to_rgb(h, s=0.8, v=0.9):
    """Convert HSV [0-1] to RGB [0-255]."""
    c = v * s
    x = c * (1 - abs((h * 6) % 2 - 1))
    m = v - c
    if h < 1/6:   r, g, b = c, x, 0
    elif h < 2/6: r, g, b = x, c, 0
    elif h < 3/6: r, g, b = 0, c, x
    elif h < 4/6: r, g, b = 0, x, c
    elif h < 5/6: r, g, b = x, 0, c
    else:         r, g, b = c, 0, x
    return (int((r + m) * 255), int((g + m) * 255), int((b + m) * 255))


def render_circle(x, y, radius, hue, img_size=96):
    """Render a colored circle on a dark background."""
    frame = np.full((img_size, img_size, 3), 25, dtype=np.uint8)
    color = hsv_to_rgb(hue)
    yy, xx = np.ogrid[:img_size, :img_size]
    mask = (xx - x) ** 2 + (yy - y) ** 2 <= radius ** 2
    frame[mask] = color
    return frame


def generate_dataset(n_episodes=400, timesteps=16, img_size=96):
    """Generate episodes with known properties."""
    rng = np.random.RandomState(42)
    all_pixels = []
    all_actions = []
    all_labels = []  # (episode, timestep, x, y, radius, hue)

    for ep in range(n_episodes):
        # Random starting state
        x = rng.uniform(20, img_size - 20)
        y = rng.uniform(20, img_size - 20)
        radius = rng.uniform(6, 16)
        hue = rng.uniform(0, 1)
        vx = rng.uniform(-3, 3)
        vy = rng.uniform(-3, 3)

        frames = []
        for t in range(timesteps):
            frames.append(render_circle(int(x), int(y), int(radius), hue, img_size))
            all_labels.append([ep, t, x / img_size, y / img_size, radius / 16, hue])
            x += vx
            y += vy
            # Bounce
            if x < radius or x > img_size - radius:
                vx = -vx
                x = np.clip(x, radius, img_size - radius)
            if y < radius or y > img_size - radius:
                vy = -vy
                y = np.clip(y, radius, img_size - radius)

        all_pixels.append(np.array(frames))
        all_actions.append(np.zeros((timesteps, 2), dtype=np.float32))

    return (
        np.array(all_pixels, dtype=np.uint8),
        np.array(all_actions, dtype=np.float32),
        np.array(all_labels, dtype=np.float32),
    )


pixels, actions, labels = generate_dataset()
print(f"Pixels: {pixels.shape}")
print(f"Labels: {labels.shape} — columns: [episode, timestep, x, y, radius, hue]")

# Show sample frames
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for row in range(2):
    for col in range(8):
        axes[row, col].imshow(pixels[row * 10, col * 2])
        axes[row, col].axis("off")
plt.suptitle("Sample frames from different episodes", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Save to HDF5 + CSV labels
with h5py.File("probe_data.h5", "w") as f:
    f.create_dataset("pixels", data=pixels, compression="gzip")
    f.create_dataset("actions", data=actions, compression="gzip")

# Save labels as CSV
with open("probe_labels.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["episode", "timestep", "x", "y", "radius", "hue"])
    for row in labels:
        writer.writerow(row.tolist())

print("Saved probe_data.h5 and probe_labels.csv")

## 2. Train the World Model

Train on this data so the encoder learns to represent the visual scene.

In [ ]:
from worldkit import WorldModel

model = WorldModel.train(
    data="probe_data.h5",
    config="nano",
    epochs=25,
    batch_size=32,
    action_dim=2,
    device="auto",
)

print(f"\nTrained: {model.num_params:,} params, latent_dim={model.latent_dim}")

## 3. Linear Probing

A **linear probe** is a Ridge regression trained on top of frozen
latent vectors. If a linear model can predict a property from the
latent vector, that information is linearly accessible in the representation.

We probe for four properties: `x`, `y`, `radius`, `hue`.

In [ ]:
result = model.probe(
    data="probe_data.h5",
    properties=["x", "y", "radius", "hue"],
    labels="probe_labels.csv",
    alpha=1.0,
    test_fraction=0.2,
)

print(result.summary)

In [ ]:
# Visualize R² scores
properties = list(result.property_scores.keys())
r2_scores = list(result.property_scores.values())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(properties, r2_scores, color=["steelblue", "coral", "mediumseagreen", "goldenrod"])
ax.set_ylabel("R² score")
ax.set_title("Linear Probing: What has the model learned?")
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)

# Add value labels
for bar, score in zip(bars, r2_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{score:.3f}", ha="center", fontsize=12)

plt.tight_layout()
plt.show()

**Interpreting the results:**
- **R² close to 1.0**: the model has learned this property well
- **R² close to 0.0**: the model doesn't encode this property (or encodes it non-linearly)
- Position (`x`, `y`) typically scores highest — it's essential for prediction
- Color (`hue`) may score lower since it doesn't affect dynamics

## 4. Encode the Full Dataset

To visualize the latent space, we first encode all frames.

In [ ]:
# Encode a subset of frames
n_frames = 2000
rng = np.random.RandomState(42)
indices = rng.choice(len(labels), size=n_frames, replace=False)

latents = []
sampled_labels = []

for idx in indices:
    ep = int(labels[idx, 0])
    t = int(labels[idx, 1])
    frame = pixels[ep, t]
    z = model.encode(frame)
    latents.append(z)
    sampled_labels.append(labels[idx])

latents = np.array(latents)  # (n_frames, latent_dim)
sampled_labels = np.array(sampled_labels)

print(f"Encoded {n_frames} frames to shape {latents.shape}")

## 5. PCA Visualization

Project the latent vectors down to 2D with PCA and color by each property.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
latents_2d = pca.fit_transform(latents)

print(f"PCA explained variance: {pca.explained_variance_ratio_[0]:.1%}, {pca.explained_variance_ratio_[1]:.1%}")

# Plot colored by each property
prop_names = ["x", "y", "radius", "hue"]
prop_indices = [2, 3, 4, 5]  # columns in labels
cmaps = ["viridis", "plasma", "coolwarm", "hsv"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name, idx, cmap in zip(axes, prop_names, prop_indices, cmaps):
    values = sampled_labels[:, idx]
    sc = ax.scatter(latents_2d[:, 0], latents_2d[:, 1], c=values,
                    cmap=cmap, s=5, alpha=0.6)
    ax.set_title(f"Colored by {name}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    plt.colorbar(sc, ax=ax)

plt.suptitle("PCA of latent space", fontsize=14)
plt.tight_layout()
plt.show()

## 6. t-SNE Visualization

t-SNE reveals non-linear structure that PCA misses.
Clusters of similar colors indicate the model groups frames
with similar properties together.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
latents_tsne = tsne.fit_transform(latents)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name, idx, cmap in zip(axes, prop_names, prop_indices, cmaps):
    values = sampled_labels[:, idx]
    sc = ax.scatter(latents_tsne[:, 0], latents_tsne[:, 1], c=values,
                    cmap=cmap, s=5, alpha=0.6)
    ax.set_title(f"Colored by {name}")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    plt.colorbar(sc, ax=ax)

plt.suptitle("t-SNE of latent space", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Probing Deeper: Prediction Accuracy Scatter

For the best-probed property, let's plot predicted vs actual values
to see how well the linear probe works.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    latents, sampled_labels[:, 2:], test_size=0.2, random_state=42
)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, (ax, name) in enumerate(zip(axes, prop_names)):
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, y_train[:, i])
    y_pred = ridge.predict(X_test)
    y_actual = y_test[:, i]

    ax.scatter(y_actual, y_pred, s=5, alpha=0.3, color="steelblue")
    # Perfect prediction line
    lim = [min(y_actual.min(), y_pred.min()), max(y_actual.max(), y_pred.max())]
    ax.plot(lim, lim, "--", color="coral", linewidth=2)
    ax.set_xlabel(f"Actual {name}")
    ax.set_ylabel(f"Predicted {name}")
    r2 = ridge.score(X_test, y_test[:, i])
    ax.set_title(f"{name} (R²={r2:.3f})")
    ax.set_aspect("equal")

plt.suptitle("Probe predictions vs ground truth", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Latent Space Structure

Let's examine the latent vector statistics — how are the dimensions used?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution of latent values
axes[0].hist(latents.flatten(), bins=100, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Latent value")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of all latent values")

# Variance per dimension
dim_variance = np.var(latents, axis=0)
axes[1].bar(range(len(dim_variance)), np.sort(dim_variance)[::-1], width=1.0, color="steelblue")
axes[1].set_xlabel("Dimension (sorted by variance)")
axes[1].set_ylabel("Variance")
axes[1].set_title("Per-dimension variance")

# Correlation matrix (subset of dims)
n_show_dims = min(32, latents.shape[1])
corr = np.corrcoef(latents[:, :n_show_dims].T)
im = axes[2].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_title(f"Correlation (first {n_show_dims} dims)")
axes[2].set_xlabel("Dimension")
axes[2].set_ylabel("Dimension")
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

print(f"Active dimensions (var > 0.01): {(dim_variance > 0.01).sum()} / {len(dim_variance)}")
print(f"Mean correlation: {np.abs(corr[np.triu_indices(n_show_dims, k=1)]).mean():.4f}")

## Next Steps

Linear probing reveals what information is **linearly accessible** in the
latent space. This is useful for:
- **Debugging**: is your model learning the right things?
- **Feature extraction**: use the latent space as a feature vector
- **Interpretability**: understand what drives model predictions

Try probing your own models:
- Generate labels from environment state (position, velocity, angle)
- Compare probing results across training epochs
- Probe with non-linear models (MLP) to find non-linearly encoded information

| Notebook | What's next |
|----------|-------------|
| [05 — Model Comparison](05_model_comparison.ipynb) | Do larger models learn more? |
| [06 — Export & Deploy](06_export_deploy.ipynb) | Export your model to ONNX |
| [07 — Plan Robot Actions](07_plan_robot_actions.ipynb) | Use the learned representations for planning |